# Energy model calibration

Fits the traction energy model the backend uses, against the Trassenfinder
samples collected by `01`, and exports the coefficients the database reads.

Successor to the exploratory notebook of the same number. That version is kept
verbatim under `exploration/2026-08_regression_exploration.ipynb`; its variant
comparison is reproduced in section 6 here, refitted on the current data, so
the record of what was tried does not depend on a stale notebook.

## What changed and why

**The data underneath moved.** The old fit ran on 448 rows collected before the
`mutter` fix, which had rejected 40 of 96 ONTD segments, biased towards short
ones. The current sample is 1,184 rows over 148 routes and 3.8-941 km.

**The functional form changed.** The old candidate was a centred OLS with an
intercept, a quadratic distance term and a distance x weight interaction. It
fits well but does not survive contact with the backend:

- it carries three **centring constants** that are properties of the training
  sample rather than of a train, with nowhere to live in the schema, and the old
  notebook's last cell died before persisting them
- its **speed coefficient is negative**, and in the backend average speed moves
  with the composition, so a faster train would be predicted to use less energy
- its intercept is a per-leg constant that does not scale with anything, and the
  backend applies the model per country leg

**The form used here** keeps what the exploration established — that the
distance x weight interaction is the dominant structure — in a shape the
pipeline can hold:

$$E = a \cdot m + d \cdot (b + c \cdot m + k \cdot m \cdot \bar{v}^2)$$

No intercept, no centring, every term proportional to something physical.

**Speed is estimated separately.** `k` comes from `01b`, the booked-speed sweep,
where route and train are held fixed. It cannot be estimated from `samples_all`,
where 99% of the variation in average speed is a property of the line rather
than of the train. See section 3.

## 1. Load

`sources/` is committed. `data/` is generated by `01` and `01b` and travels
through Drive; `ensure_local()` fetches it on first use.

The speed sweep is optional. Without it `k` is zero and the model does not
respond to speed — which is what the backend does today, only with the rest of
the model correct.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

from data_sources import DATA_DIR, SEED_DIR, ensure_local

SEED_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE = "samples_all"  # or "samples_ontd" / "samples_synthetic" to fit one source


# Which speed the drag term is fitted against. This must match what the BACKEND
# supplies, not what is physically correct, and those are different things.
#
# Drag responds to the distance-weighted RMS speed: a route with a fast section
# and slow approaches burns more than one held steady at the same average. The
# sweep measures that gap at 8-9% of the drag term on typical routes and up to
# 41% on the most variable ones.
#
# But CountryLeg carries only the average, so the deployed model is applied to
# the average. Fitting on RMS would yield the true physical coefficient and then
# under-predict by the convexity gap every time the backend used it. Fitting on
# the average instead lets the coefficient absorb the median convexity of the
# calibration routes, which is self-consistent: what is left unexplained is the
# spread between routes, not a systematic offset.
#
# So "average" is the default and the one to ship. "rms" is for reading the true
# physical coefficient and for measuring what the choice costs.
DRAG_SPEED = "average"  # "average" | "rms"


def drag_speed(df):
    """Speed the drag term is fitted and applied against — see DRAG_SPEED."""
    if DRAG_SPEED == "rms" and "v_rms_kmh" in df:
        return df["v_rms_kmh"].where(df["v_rms_kmh"] > 0, df["avg_speed_kmh"])
    return df["avg_speed_kmh"]


def add_derived(df):
    """Columns every stage below needs. avg_speed_kmh is the realised average,
    the same quantity the backend has on CountryLeg — not the booked v_max."""
    df = df.copy()
    df["segment_id"] = (
        df["route_name"].astype(str)
        + "__"
        + df["start_ds100"].astype(str)
        + "__"
        + df["end_ds100"].astype(str)
    )
    df["avg_speed_kmh"] = df["distance_km"] / (df["travel_time_min"] / 60)
    df["m"] = df["weight_t"]
    df["d"] = df["distance_km"]
    df["md"] = df["m"] * df["d"]
    df["v_drag_kmh"] = drag_speed(df)
    df["mdv2"] = df["md"] * df["v_drag_kmh"] ** 2
    if "length_m" in df.columns:
        df["ldv2"] = df["length_m"] * df["d"] * df["avg_speed_kmh"] ** 2
    df["hours"] = df["travel_time_min"] / 60.0

    # Traction is what responds to speed. Auxiliaries respond to time. Samples
    # collected before 2026-08-30 carry only the total, in which case the split
    # is unavailable and the fit falls back to modelling the total directly.
    # The per-route-point energy fields are cumulative running totals, so 01
    # takes the last point rather than summing. A sample collected with the
    # summing bug reads roughly 22x too high at the median; the guard below
    # catches it rather than fitting it.
    if "energy_traktion_kwh" in df.columns:
        df["energy_aux_kwh"] = df["energy_hilfsbetriebe_kwh"].fillna(0) + df[
            "energy_wagen_kwh"
        ].fillna(0)
        df["energy_fit_kwh"] = df["energy_traktion_kwh"]
    else:
        df["energy_aux_kwh"] = 0.0
        df["energy_fit_kwh"] = df["energy_kwh"]
    return df


data = add_derived(pd.read_csv(ensure_local(f"{SAMPLE}.csv")))

SPLIT_AVAILABLE = "energy_traktion_kwh" in data.columns

if SPLIT_AVAILABLE:
    # Sanity-gate the split before anything is fitted on it.
    parts = (
        data["energy_traktion_kwh"]
        + data["energy_hilfsbetriebe_kwh"]
        + data["energy_wagen_kwh"]
    )
    overshoot = (parts / data["energy_kwh"]).median()
    if overshoot > 1.05:
        raise ValueError(
            f"The energy components sum to {overshoot:.1f}x the trip total. "
            "They are cumulative per-route-point running totals and this sample "
            "was collected by summing them instead of taking the last point. "
            "Re-run 01 with trassenfinder.py from 2026-08-30 or later."
        )
    print("energy split available - fitting traction, auxiliaries modelled on time")
else:
    print(
        "NO energy split in this sample - it predates 2026-08-30. The fit will "
        "model the total, so the speed term absorbs the time-dependent "
        "auxiliary load. Re-run 01 to collect the split."
    )

print(
    f"{SAMPLE}: {len(data)} rows, {data['segment_id'].nunique()} routes, "
    f"{data['composition_id'].nunique()} compositions"
)
print(
    data[["m", "d", "avg_speed_kmh", "energy_kwh"]]
    .describe()
    .loc[["min", "50%", "max"]]
    .round(1)
    .to_string()
)

# --- the speed sweep, if it has been collected --------------------------------
speed_path = DATA_DIR / "samples_speed.csv"
if speed_path.is_file():
    speed = add_derived(pd.read_csv(speed_path))
    print(
        f"\nsamples_speed: {len(speed)} rows, "
        f"{speed['segment_id'].nunique()} segments, "
        f"grid {sorted(speed['v_max_requested_kmh'].unique())}"
    )
else:
    speed = None
    print(
        "\nsamples_speed.csv not present — run 01b_speed_sweep.ipynb to "
        "estimate the speed coefficient. k will be held at 0."
    )

energy split available - fitting traction, auxiliaries modelled on time
samples_all: 1192 rows, 149 routes, 8 compositions
         m      d  avg_speed_kmh  energy_kwh
min  313.0    3.8           37.3        30.0
50%  516.8  158.5          121.5      1389.0
max  636.0  932.2          179.4     11185.0

samples_speed: 1440 rows, 26 segments, grid [np.int64(60), np.int64(90), np.int64(120), np.int64(150), np.int64(180), np.int64(200), np.int64(230)]


## 2. Mass basis

The sample's `weight_t` is the **Wagenzugmasse**: coaches at 80% load,
**excluding** the 87 t locomotive. That is what `01` sends as
`wagenzugmasse_t`, with the locomotive specified separately by
`triebfahrzeug.hauptnummer`.

This matters for the handover. The backend has both quantities:

| | |
|---|---|
| `Composition.total_weight_t()` | coaches only — **matches the calibration** |
| `Composition.total_gross_weight_t()` | coaches + locomotives |

Fit and application must use the same basis. The locomotive's own contribution
is inside `b`, the mass-independent per-km term, so applying these coefficients
against gross mass would count it twice — around +11% on a 500 km leg.

The cell below asserts the basis rather than trusting the column name.

In [2]:
comp_ref = pd.read_csv(DATA_DIR.parent / "sources" / "compositions.csv").set_index(
    "composition_id"
)

check = (
    data.groupby("composition_id")["m"]
    .first()
    .to_frame("sample_weight_t")
    .join(
        comp_ref[
            [
                "coaches_gross_weight_80pct_t_wagenzugmasse",
                "total_weight_incl_loco_80pct_t",
                "loco_weight_t",
            ]
        ]
    )
)
check["matches_wagenzugmasse"] = np.isclose(
    check["sample_weight_t"], check["coaches_gross_weight_80pct_t_wagenzugmasse"]
)
print(check.round(1).to_string())

assert check["matches_wagenzugmasse"].all(), (
    "weight_t is not the Wagenzugmasse — the mass basis has changed and the "
    "backend handover note in section 8 no longer applies."
)
print("\nMass basis confirmed: coaches at 80% load, locomotive excluded.")

                sample_weight_t  coaches_gross_weight_80pct_t_wagenzugmasse  total_weight_incl_loco_80pct_t  loco_weight_t  matches_wagenzugmasse
composition_id                                                                                                                                   
NEW-BAL-14                627.6                                       627.6                           714.6           87.0                   True
NEW-BAL-7                 313.8                                       313.8                           400.8           87.0                   True
REF-BAL-9                 501.4                                       501.4                           588.4           87.0                   True
REF-BUD-12                636.0                                       636.0                           723.0           87.0                   True
REF-BUD-6                 323.3                                       323.3                           410.3           87.0  

## 3. Stage 1 — the speed coefficient, from the sweep only

`k` is estimated **within** `(segment, composition)` groups. Inside a group the
route, the station pair and the train are all fixed, so the only thing that
moves is the booked speed. Demeaning by group removes everything constant and
leaves the speed response alone.

This is the whole point of the two-stage design. Estimating `k` on
`samples_all` instead would identify it from variation *between* routes, where
low average speed marks a branch line or a station approach — which costs energy
for reasons that have nothing to do with drag. That is where the negative
coefficient in the old notebook came from.

A group is `(segment, composition, stratum)`. The stratum belongs in the key
because `schnellfahrstrecken_meiden` changes which *line* the train runs on:
with high-speed lines avoided the booked speed stops binding around 120 km/h,
and allowing them is what makes 120-200 reachable at all. Estimating `k`
separately in each stratum gives two readings over two speed ranges, which is
worth more than either alone.

Only route-stable groups are used: if `distance_km` moved across the grid, the
optimiser changed the path and the group is not a controlled experiment.

The regressor is $m \cdot d \cdot \bar{v}^2$ using **realised** average speed,
not booked `v_max`, because realised average speed is what the backend has on
`CountryLeg`. Grid points where the booked speed did not bind contribute
nothing, which is the correct treatment rather than something to filter out.

In [3]:
K_SPEED = 0.0
k_fit = None

# A group is one route run by one train under one avoidance setting. The stratum
# belongs in the key: schnellfahrstrecken_meiden changes which line the train is
# on, so mixing strata inside a group would put a route change back into the
# speed response. Older sweeps have no stratum column and are all conventional.
GROUP_KEYS = ["segment_id", "composition_id", "stratum"]


def fit_k(df, label):
    """Within-group estimate of the speed coefficient. Returns (k, result)."""
    grp = df.groupby(GROUP_KEYS)
    y = df["energy_fit_kwh"] - grp["energy_fit_kwh"].transform("mean")
    x = df["mdv2"] - grp["mdv2"].transform("mean")
    res = sm.OLS(y, x.to_frame("mdv2")).fit(
        cov_type="cluster", cov_kwds={"groups": df["segment_id"]}
    )
    k = float(res.params["mdv2"])
    lo, hi = res.conf_int().loc["mdv2"]
    print(
        f"  {label:14s} k = {k:.4e}   95% CI [{lo:.3e}, {hi:.3e}]   "
        f"p = {res.pvalues['mdv2']:.3g}   groups = {grp.ngroups}   "
        f"within R^2 = {res.rsquared:.4f}"
    )
    return k, res


if speed is not None:
    if "stratum" not in speed.columns:
        speed["stratum"] = "conventional"

    stab = speed.groupby(GROUP_KEYS)["d"].agg(["min", "max"])
    stable_keys = stab.index[(stab["max"] - stab["min"]) / stab["min"] < 0.005]

    sweep = speed[speed.set_index(GROUP_KEYS).index.isin(set(stable_keys))].copy()

    # Groups need at least two distinct realised speeds to say anything.
    usable = sweep.groupby(GROUP_KEYS)["avg_speed_kmh"].transform("nunique")
    sweep = sweep[usable >= 2]

    print(f"route-stable groups: {len(stable_keys)} of {len(stab)}")
    print(
        f"usable rows:         {len(sweep)} in "
        f"{sweep.groupby(GROUP_KEYS).ngroups} groups"
    )
    print(f"strata present:      {sorted(sweep['stratum'].unique())}")

    if "v_rms_kmh" in sweep.columns and (sweep["v_rms_kmh"] > 0).any():
        prof = sweep[sweep["v_rms_kmh"] > 0]
        conv = ((prof["v_rms_kmh"] / prof["avg_speed_kmh"]) ** 2 - 1) * 100
        print(
            f"\nconvexity gap: median {conv.median():.1f}%, "
            f"90th pct {conv.quantile(0.9):.1f}%, max {conv.max():.1f}% "
            "of the drag term"
        )
        print(
            f"  Fitting on {DRAG_SPEED} speed. With 'average' the "
            "coefficient absorbs the median, and the residual is the spread "
            "between routes, not a systematic offset. Set DRAG_SPEED='rms' "
            "to read the true physical coefficient instead."
        )
        print(f"  peak speed actually reached: {prof['v_peak_kmh'].max():.0f} km/h")
    print(
        f"realised speed range: {sweep['avg_speed_kmh'].min():.1f} - "
        f"{sweep['avg_speed_kmh'].max():.1f} km/h"
    )
    if "segment_set" in sweep.columns:
        print(
            sweep.groupby("segment_set")["avg_speed_kmh"]
            .agg(["min", "max", "size"])
            .round(1)
            .to_string()
        )
    print(
        "\nThis is the range the drag term is identified over. Beyond it the "
        "v^2 term extrapolates, and drag is quadratic, so error grows fast. "
        "Recorded in energy_calibration_meta.csv as drag_identified_up_to_kmh "
        "so the backend can flag a leg that exceeds it.\n"
    )

    # Per stratum first. The conventional network caps the train around 120 km/h
    # booked; the sfs stratum is what reaches 120-200. Two estimates over two
    # speed ranges either agree, which is evidence the v^2 form holds across the
    # range, or disagree, which is a finding rather than a nuisance.
    #
    # If they disagree, suspect tunnels before suspecting the form: NBS are
    # tunnel-heavy and tunnel drag is materially higher than open-air drag, so
    # an sfs estimate above the conventional one has a mundane explanation.
    per_stratum = {}
    for name, part in sweep.groupby("stratum"):
        if part.groupby(GROUP_KEYS).ngroups >= 2:
            per_stratum[name] = fit_k(part, name)

    print()
    K_SPEED, k_fit = fit_k(sweep, "pooled")

    if len(per_stratum) > 1:
        ks = {n: k for n, (k, _) in per_stratum.items()}
        spread = (
            max(ks.values()) / min(ks.values())
            if min(ks.values()) > 0
            else float("nan")
        )
        print(f"\nstratum estimates differ by a factor of {spread:.2f}")
        if spread > 1.5:
            print(
                "  They do not agree. Do not ship the pooled number as if it "
                "were one coefficient: report both, and decide deliberately "
                "which range the backend operates in."
            )
        else:
            print("  Close enough to treat as one coefficient over the range.")

    if K_SPEED <= 0:
        print(
            "\nWARNING: k is not positive. Either the booked speed never bound "
            "(check section 7 of 01b — realised speed should move with the grid) "
            "or auxiliary consumption at low speed dominates drag over this "
            "range. Do NOT ship a negative k; hold it at 0 and record why."
        )

    # What the term is worth at a realistic operating point.
    m_ref, d_ref, v_ref = 501.4, 500.0, 95.0
    print(
        f"\nAt {m_ref:.0f} t, {d_ref:.0f} km, {v_ref:.0f} km/h the speed term "
        f"contributes {K_SPEED * m_ref * d_ref * v_ref**2:.0f} kWh"
    )
else:
    print("No sweep — k held at 0.0. The model will not respond to speed.")

route-stable groups: 199 of 225
usable rows:         1274 in 199 groups
strata present:      ['conventional', 'sfs']

convexity gap: median 7.7%, 90th pct 19.5%, max 41.5% of the drag term
  Fitting on average speed. With 'average' the coefficient absorbs the median, and the residual is the spread between routes, not a systematic offset. Set DRAG_SPEED='rms' to read the true physical coefficient instead.
  peak speed actually reached: 220 km/h
realised speed range: 55.9 - 165.5 km/h
              min    max  size
segment_set                   
nbs          56.7  165.5   602
sampled      55.9  150.6   672

This is the range the drag term is identified over. Beyond it the v^2 term extrapolates, and drag is quadratic, so error grows fast. Recorded in energy_calibration_meta.csv as drag_identified_up_to_kmh so the backend can flag a leg that exceeds it.

  conventional   k = 9.2906e-07   95% CI [9.062e-07, 9.520e-07]   p = 0   groups = 122   within R^2 = 0.9347
  sfs            k = 7.7338e

### Which does drag scale with — mass, or length?

The model form carries `k · m · v̄²`, tying aerodynamic drag to mass. That is not
the physics. Drag comes from frontal area and from skin friction along the
train's length; **mass belongs in rolling resistance and acceleration.**

Across the refurbished fleet the two cannot be told apart, because mass and
length move together at a near-constant 1.98–2.11 t/m. The new-build
compositions sit at 1.69 t/m and break the collinearity, which is why the sweep
covers two mass-matched pairs.

Identified in two stages. Within a group the route, the train and the stratum
are all fixed, so a regression of energy on $\bar{v}^2$ gives that group's drag
slope directly. Divided by distance, that is a drag coefficient per kilometre,
and it can then be regressed **across** groups on mass and on length — which is
the comparison the within-group estimator cannot make.

`DRAG_FORM` decides what is adopted. `"auto"` takes the length form only if it
beats the mass form on second-stage fit and its length coefficient is
significant; set `"mass"` or `"length"` to force one.

In [4]:
DRAG_FORM = "auto"  # "auto" | "mass" | "length"

K_MASS, K_LEN_CONST, K_LEN_SLOPE = K_SPEED, 0.0, 0.0
ADOPTED_FORM = "mass"
K_M_DRAG = 0.0
K_L_DRAG = 0.0

if speed is not None and "length_m" in sweep.columns:
    # --- stage 1: one drag slope per group -------------------------------
    slopes = []
    for keys, g in sweep.groupby(GROUP_KEYS):
        if g["avg_speed_kmh"].nunique() < 3:
            continue
        x = sm.add_constant(g["v_drag_kmh"] ** 2)
        fit = sm.OLS(g["energy_fit_kwh"], x).fit()
        slopes.append(
            {
                "composition_id": keys[1],
                "drag_per_km": fit.params.iloc[1] / g["d"].iloc[0],
                "m": g["m"].iloc[0],
                "L": g["length_m"].iloc[0],
                "segment_id": keys[0],
            }
        )

    drag = pd.DataFrame(slopes)
    print(f"{len(drag)} groups with enough speed points for a slope\n")
    print("Median drag coefficient per km, by composition:")
    print(
        drag.groupby("composition_id")
        .agg(m=("m", "first"), L=("L", "first"), drag_per_km=("drag_per_km", "median"))
        .assign(t_per_m=lambda d: (d["m"] / d["L"]).round(2))
        .round(6)
        .to_string()
    )

    # --- stage 2: what does that coefficient scale with? ------------------
    cluster = {"cov_type": "cluster", "cov_kwds": {"groups": drag["segment_id"]}}
    mass_form = sm.OLS(drag["drag_per_km"], drag[["m"]]).fit(**cluster)
    length_form = sm.OLS(drag["drag_per_km"], sm.add_constant(drag[["L"]])).fit(
        **cluster
    )

    print(
        f"\n  mass form   drag/km = k*m          R^2 = {mass_form.rsquared:.4f}"
        f"   k = {mass_form.params['m']:.4e}"
    )
    print(
        f"  length form drag/km = k1 + k2*L   R^2 = {length_form.rsquared:.4f}"
        f"   k1 = {length_form.params['const']:.4e}"
        f"   k2 = {length_form.params['L']:.4e}"
        f"   p(k2) = {length_form.pvalues['L']:.4g}"
    )

    K_MASS = float(mass_form.params["m"])
    K_LEN_CONST = float(length_form.params["const"])
    K_LEN_SLOPE = float(length_form.params["L"])

    # --- adopted: combined nonnegative form k_m*m + k_L*L -----------------
    # Physics has both: frontal/bogie effects that ride with train size, and
    # skin friction that rides with length. The single-variable forms above
    # are diagnostics; the model ships the combined fit, constrained
    # nonnegative because a negative drag component is a fitting artifact,
    # never a mechanism.
    from scipy.optimize import nnls

    X_c = drag[["m", "L"]].to_numpy()
    y_c = drag["drag_per_km"].to_numpy()
    (K_M_DRAG, K_L_DRAG), _ = nnls(X_c, y_c)
    comb_pred = X_c @ [K_M_DRAG, K_L_DRAG]
    ss = 1 - ((y_c - comb_pred) ** 2).sum() / ((y_c - y_c.mean()) ** 2).sum()
    print(
        f"\n  combined    drag/km = k_m*m + k_L*L   R^2 = {ss:.4f}   "
        f"k_m = {K_M_DRAG:.4e}   k_L = {K_L_DRAG:.4e}"
    )

    # --- the mass-matched pairs decide it ---------------------------------
    # R^2 is useless here: drag per km tracks train size under either form, so
    # both fit well and the margin between them is noise. The pairs are the
    # designed contrast — same mass, 17% more length — and under the mass form
    # they must show no gap at all.
    print("\nMass-matched pairs (same mass, +17% length):")
    gaps = []
    for ref_id, new_id in (("REF-COUCH-6", "NEW-BAL-7"), ("REF-BUD-12", "NEW-BAL-14")):
        pair = drag[drag["composition_id"].isin([ref_id, new_id])]
        if pair["composition_id"].nunique() < 2:
            print(f"  {ref_id} / {new_id}: not both in the sweep")
            continue
        med = pair.groupby("composition_id")["drag_per_km"].median()
        gap = (med[new_id] / med[ref_id] - 1) * 100
        gaps.append(gap)
        print(f"  {new_id} vs {ref_id}: {gap:+.1f}% drag per km")

    if gaps:
        median_gap = float(np.median(gaps))
        print(
            f"  median pair gap: {median_gap:+.1f}% against a +17.2% length "
            "difference and no mass difference"
        )
    else:
        median_gap = 0.0
        print("  no complete pair in the sweep - the forms cannot be separated")

    if DRAG_FORM == "auto":
        # Both conditions, because either alone is weak: a gap with an
        # insignificant coefficient is noise, and a significant coefficient
        # with no gap is collinearity being read as an effect.
        gap_real = abs(median_gap) > 5.0
        significant = length_form.pvalues["L"] < 0.05
        ADOPTED_FORM = "length" if (gap_real and significant) else "mass"
        print(
            f"\n  auto -> {ADOPTED_FORM} "
            f"(pair gap over 5%: {gap_real}, k2 significant: {significant})"
        )
    else:
        ADOPTED_FORM = DRAG_FORM
        print(f"\n  forced -> {ADOPTED_FORM}")

    if ADOPTED_FORM == "length":
        print(
            "  Drag does not scale with mass. A coefficient fitted on "
            "refurbished stock and applied to new-build stock is biased by "
            "roughly the pair gap, on exactly the comparison the tool exists "
            "to make."
        )
else:
    print("No sweep, or no length column - drag form not testable, mass assumed.")


def drag_coefficient(df):
    """Drag per km per (km/h)^2 - the combined form the model ships."""
    if K_M_DRAG or K_L_DRAG:
        return K_M_DRAG * df["m"] + K_L_DRAG * df["length_m"]
    return K_MASS * df["m"]

199 groups with enough speed points for a slope

Median drag coefficient per km, by composition:
                    m      L  drag_per_km  t_per_m
composition_id                                    
NEW-BAL-14      627.6  371.4     0.000596     1.69
NEW-BAL-7       313.8  185.7     0.000330     1.69
REF-BAL-9       501.4  237.6     0.000439     2.11
REF-BUD-12      636.0  316.8     0.000558     2.01
REF-COUCH-6     313.0  158.4     0.000301     1.98

  mass form   drag/km = k*m          R^2 = 0.9409   k = 8.9309e-07
  length form drag/km = k1 + k2*L   R^2 = 0.5167   k1 = 7.0577e-05   k2 = 1.4226e-06   p(k2) = 7.769e-83

  combined    drag/km = k_m*m + k_L*L   R^2 = 0.5154   k_m = 4.3017e-07   k_L = 8.7132e-07

Mass-matched pairs (same mass, +17% length):
  NEW-BAL-7 vs REF-COUCH-6: +9.8% drag per km
  NEW-BAL-14 vs REF-BUD-12: +6.8% drag per km
  median pair gap: +8.3% against a +17.2% length difference and no mass difference

  auto -> length (pair gap over 5%: True, k2 significant: T

## 4. Stage 2 — level coefficients on the main sample

With `k` fixed, subtract the speed term from the response and fit the rest by
OLS through the origin:

$$E - k \cdot m \cdot d \cdot \bar{v}^2 = a \cdot m + b \cdot d + c \cdot m \cdot d$$

Fixing `k` rather than fitting it jointly is deliberate: a free `k` here would
be identified from between-route variation again and would drag the level terms
with it.

Standard errors are clustered on `segment_id`, since the eight compositions on
one segment are eight readings of the same route.

`a` is small and usually insignificant. It is kept because the backend applies
the model **per country leg**, so a per-leg constant is the right place to carry
start-up and terminal energy. Section 8 quantifies what it costs on a
multi-country trip.

In [5]:
def fit_levels(df, k):
    """Fit a, b, c against traction energy, with the drag term held fixed.

    k is retained for the mass form; under the length form the drag term is
    computed from drag_coefficient() instead, so the offset is the same shape
    either way and the level terms are unaffected by which was adopted.
    """
    if (K_M_DRAG or K_L_DRAG) and "length_m" in df.columns:
        drag_offset = drag_coefficient(df) * df["d"] * drag_speed(df) ** 2
    else:
        drag_offset = k * df["mdv2"]
    y = df["energy_fit_kwh"] - drag_offset

    X = df[["m", "d", "md"]]
    return sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": df["segment_id"]})


levels = fit_levels(data, K_SPEED)

# A per-leg constant must be nonnegative: it carries start/stop energy. A
# negative estimate is collinearity, not physics - clamp to zero and refit
# so the other terms are not distorted by an impossible offset.
A_CLAMPED = False
if float(levels.params["m"]) < 0:
    A_CLAMPED = True
    print(f"a = {float(levels.params['m']):.4f} < 0 -> clamped to 0, refit")

if A_CLAMPED:
    drag_off = (
        drag_coefficient(data) * data["d"] * drag_speed(data) ** 2
        if (K_M_DRAG or K_L_DRAG)
        else K_SPEED * data["mdv2"]
    )
    refit = sm.OLS(data["energy_fit_kwh"] - drag_off, data[["d", "md"]]).fit(
        cov_type="cluster", cov_kwds={"groups": data["segment_id"]}
    )
    print(refit.summary().tables[1])
    A = 0.0
    B, Ck = (float(refit.params[n]) for n in ("d", "md"))
else:
    A, B, Ck = (float(levels.params[n]) for n in ("m", "d", "md"))

print(levels.summary().tables[1])
print(f"\na = {A:.6f} kWh/t          per-leg constant, scales with mass")
print(f"b = {B:.4f} kWh/km          per-km, mass-independent (incl. the loco)")
print(f"c = {Ck:.6f} kWh/(t*km)   per tonne-km")
print(f"k = {K_SPEED:.6e} kWh/(t*km*(km/h)^2)   from the sweep")


def predict(df, a=None, b=None, c=None, k=None):
    a = A if a is None else a
    b = B if b is None else b
    c = Ck if c is None else c
    k = K_SPEED if k is None else k
    if (K_M_DRAG or K_L_DRAG) and "length_m" in df.columns:
        drag = drag_coefficient(df)
    else:
        drag = k * df["m"]
    return a * df["m"] + df["d"] * (b + c * df["m"] + drag * drag_speed(df) ** 2)


data["predicted_traction_kwh"] = predict(data)
print(
    f"\ntraction in-sample R^2 = "
    f"{r2_score(data['energy_fit_kwh'], data['predicted_traction_kwh']):.4f}"
)

a = -0.0285 < 0 -> clamped to 0, refit
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
d              1.3304      0.024     56.542      0.000       1.284       1.377
md             0.0013      0.000      4.393      0.000       0.001       0.002
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
m             -0.0285      0.071     -0.402      0.688      -0.167       0.110
d              1.3305      0.024     56.542      0.000       1.284       1.377
md             0.0014      0.000      3.809      0.000       0.001       0.002

a = 0.000000 kWh/t          per-leg constant, scales with mass
b = 1.3304 kWh/km          per-km, mass-independent (incl. the loco)
c = 0.001336 kWh/(t*km)   per tonne-km
k = 8.411943e-07 kWh/(t*km*(km/h)^2)   from the sweep

traction in-sample R^2 

### Auxiliary power

Traction is not the whole bill. Locomotive auxiliaries and, when the request
enables it, coach hotel load draw power for as long as the train is moving, so
they scale with **time**, not with the square of speed.

That distinction is why the split matters. Fitted as one number, the speed term
has to absorb a component that gets *larger* as the train goes slower, which
bends the low end of the curve the wrong way and was the source of the
turning-point worry in earlier runs.

Modelled here as a constant draw:

$$E_{aux} = P_{aux} \cdot t_h$$

**Coach hotel load is currently zero by construction.** The request sets
`zusaetzlicher_energieverbrauch_pro_wagen_kw` to 0, so `energy_wagen_kwh` comes
back empty and only the locomotive's own auxiliaries are counted. For a night
train with sleeping cars that is a real omission — heating, air conditioning and
lighting run all night — and setting it is a modelling decision, not something
to patch in quietly. Until it is taken, the auxiliary term here is a floor.

In [6]:
P_AUX_KW = 0.0

if SPLIT_AVAILABLE and data["energy_aux_kwh"].sum() > 0:
    aux = data[data["hours"] > 0].copy()
    aux["kw"] = aux["energy_aux_kwh"] / aux["hours"]

    print("Implied auxiliary draw, kW:")
    print(aux["kw"].describe().round(1).to_string())

    # Through the origin: energy proportional to time, with no standing
    # component, since a standing train is not in the sample.
    aux_fit = sm.OLS(aux["energy_aux_kwh"], aux[["hours"]]).fit(
        cov_type="cluster", cov_kwds={"groups": aux["segment_id"]}
    )
    P_AUX_KW = float(aux_fit.params["hours"])
    lo, hi = aux_fit.conf_int().loc["hours"]
    print(
        f"\nP_aux = {P_AUX_KW:.1f} kW   95% CI [{lo:.1f}, {hi:.1f}]   "
        f"R^2 = {aux_fit.rsquared:.4f}"
    )

    print(
        "\nMedian implied kW by composition (flat means it is a "
        "locomotive-side load, not a per-coach one):"
    )
    print(aux.groupby("composition_id")["kw"].median().round(1).to_string())

    share = data["energy_aux_kwh"].sum() / data["energy_kwh"].sum() * 100
    print(f"\nAuxiliaries are {share:.1f}% of total energy across the sample.")
    if data["energy_wagen_kwh"].sum() == 0:
        print(
            "  energy_wagen_kwh is zero throughout: coach hotel load was "
            "switched off in the request. Treat this share as a floor."
        )
else:
    print("No auxiliary split in this sample - P_aux held at 0.")


def predict_total(df):
    """Traction plus auxiliaries, which is what the backend prices."""
    return predict(df) + P_AUX_KW * df["hours"]


data["predicted_kwh"] = predict_total(data)
print(
    f"\ntotal-energy in-sample R^2 = "
    f"{r2_score(data['energy_kwh'], data['predicted_kwh']):.4f}"
)

Implied auxiliary draw, kW:
count    1192.0
mean       99.5
std         1.9
min        84.0
25%        99.4
50%        99.9
75%       100.0
max       108.0

P_aux = 99.9 kW   95% CI [99.9, 99.9]   R^2 = 1.0000

Median implied kW by composition (flat means it is a locomotive-side load, not a per-coach one):
composition_id
NEW-BAL-14      99.9
NEW-BAL-7       99.9
REF-BAL-9       99.9
REF-BUD-12      99.8
REF-BUD-6       99.9
REF-COUCH-10    99.9
REF-COUCH-6     99.9
REF-PREM-12     99.8

Auxiliaries are 9.1% of total energy across the sample.
  energy_wagen_kwh is zero throughout: coach hotel load was switched off in the request. Treat this share as a floor.

total-energy in-sample R^2 = 0.9637


## 5. Validation

Held out **by route**, not by row. The eight compositions on one segment are
near-duplicates, so a random row split would put the same route on both sides
and report a fit that does not exist.

Three things have to hold, in this order of importance:

1. **Per-composition bias.** Comparing compositions is what the tool is for. A
   model that misprices light trains against heavy ones is worse than useless
   here even if its overall error is small.
2. **Bias across the distance range**, since country legs run from a few km to
   over a thousand.
3. Overall error.

In [7]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(data, groups=data["segment_id"]))
train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
assert set(train["segment_id"]).isdisjoint(set(test["segment_id"]))

held = fit_levels(train, K_SPEED)
a_h, b_h, c_h = (float(held.params[n]) for n in ("m", "d", "md"))
test["pred_traction"] = predict(test, a_h, b_h, c_h)
test["pred"] = test["pred_traction"] + P_AUX_KW * test["hours"]

mae = mean_absolute_error(test["energy_kwh"], test["pred"])
rmse = float(np.sqrt(((test["energy_kwh"] - test["pred"]) ** 2).mean()))
rel = (test["energy_kwh"] - test["pred"]).abs() / test["energy_kwh"] * 100

print(f"held-out routes: {test['segment_id'].nunique()}, rows: {len(test)}")
print(f"  MAE   {mae:8.1f} kWh")
print(f"  RMSE  {rmse:8.1f} kWh")
print(f"  R^2   {r2_score(test['energy_kwh'], test['pred']):8.4f}")
print(
    f"  median relative error {rel.median():.1f}%   90th pct {rel.quantile(0.9):.1f}%"
)

# --- 1. per composition, fleet-weighted --------------------------------------
data["pred_full"] = predict_total(data)
per_comp = data.groupby("composition_id").apply(
    lambda x: pd.Series(
        {
            "weight_t": x["m"].iloc[0],
            "observed_kwh_km": x["energy_kwh"].sum() / x["d"].sum(),
            "model_kwh_km": x["pred_full"].sum() / x["d"].sum(),
        }
    ),
    include_groups=False,
)
per_comp["bias_pct"] = (
    per_comp["model_kwh_km"] / per_comp["observed_kwh_km"] - 1
) * 100
print("\nFleet-weighted intensity by composition")
print(per_comp.sort_values("weight_t").round(2).to_string())
print(
    f"\nobserved spread heaviest/lightest: "
    f"{per_comp['observed_kwh_km'].max() / per_comp['observed_kwh_km'].min():.3f}"
)
print(
    f"model spread:                      "
    f"{per_comp['model_kwh_km'].max() / per_comp['model_kwh_km'].min():.3f}"
)

# --- 2. across the distance range --------------------------------------------
test["band"] = pd.cut(test["d"], [0, 50, 100, 200, 400, 1000])
per_band = test.groupby("band", observed=True).apply(
    lambda x: pd.Series(
        {
            "n": len(x),
            "median_rel_pct": (
                (x["energy_kwh"] - x["pred"]).abs() / x["energy_kwh"] * 100
            ).median(),
            "mean_bias_pct": (
                (x["pred"] - x["energy_kwh"]) / x["energy_kwh"] * 100
            ).mean(),
        }
    ),
    include_groups=False,
)
print("\nHeld-out error by distance band")
print(per_band.round(1).to_string())

# --- poolability, re-tested rather than quoted ------------------------------
# calib/README.md carried a poolability result from a 2026-08-24 run whose data
# has since been superseded twice. Automated here so the claim always has the
# current data behind it.
if data["source"].nunique() > 1:
    pool = data.assign(is_syn=(data["source"] == "synthetic").astype(float))
    pool["syn_d"] = pool["is_syn"] * pool["d"]
    pool_fit = sm.OLS(
        pool["energy_fit_kwh"] - K_SPEED * pool["mdv2"],
        pool[["m", "d", "md", "syn_d"]],
    ).fit(cov_type="cluster", cov_kwds={"groups": pool["segment_id"]})
    print(
        f"\nPoolability: source indicator on the per-km term, "
        f"p = {pool_fit.pvalues['syn_d']:.4f} "
        f"({pool_fit.params['syn_d']:+.4f} kWh/km)"
    )
    print(
        "  Not significant means the two route sources describe the same "
        "train and pooling is sound. Significant means they do not, and "
        "calib/README.md needs updating rather than re-quoting."
    )

# --- monotonicity, the property the old candidate lost -----------------------
probe = pd.DataFrame(
    {"m": [313.0, 501.4, 636.0], "d": [500.0] * 3, "avg_speed_kmh": [95.0] * 3}
)
probe["hours"] = probe["d"] / probe["avg_speed_kmh"]
probe["kwh"] = predict_total(probe)
probe["kwh_km"] = probe["kwh"] / probe["d"]
print("\nMonotone in mass at 500 km, 95 km/h:")
print(probe.round(2).to_string(index=False))
assert probe["kwh"].is_monotonic_increasing, "heavier train predicted to use less"

if K_SPEED > 0:
    probe_v = pd.DataFrame(
        {
            "m": [501.4] * 4,
            "d": [500.0] * 4,
            "avg_speed_kmh": [70.0, 90.0, 110.0, 130.0],
        }
    )
    probe_v["hours"] = probe_v["d"] / probe_v["avg_speed_kmh"]
    probe_v["kwh"] = predict_total(probe_v)
    print("\nMonotone in speed at 501 t, 500 km:")
    print(probe_v.round(1).to_string(index=False))
    assert probe_v["kwh"].is_monotonic_increasing, "faster train predicted to use less"

held-out routes: 30, rows: 240
  MAE      259.1 kWh
  RMSE     425.0 kWh
  R^2     0.9465
  median relative error 10.5%   90th pct 45.1%

Fleet-weighted intensity by composition
                weight_t  observed_kwh_km  model_kwh_km  bias_pct
composition_id                                                   
REF-COUCH-6        313.0             6.77          6.81      0.50
NEW-BAL-7          313.8             7.13          7.17      0.66
REF-BUD-6          323.3             6.87          6.89      0.26
REF-BAL-9          501.4             9.27          9.27     -0.02
REF-COUCH-10       532.2             9.76          9.73     -0.32
NEW-BAL-14         627.6            11.51         11.69      1.58
REF-PREM-12        631.6            11.08         11.07     -0.06
REF-BUD-12         636.0            11.13         11.10     -0.22

observed spread heaviest/lightest: 1.698
model spread:                      1.717

Held-out error by distance band
                n  median_rel_pct  mean_bias_p

## 6. What was tried and rejected

Kept so the next person does not repeat it. All refitted on the current sample
and scored on the same held-out routes, so the comparison is like for like.

The exploration's finding stands: the distance x weight interaction is the
dominant structure, and it is the term carried forward as `c`. What does not
carry forward is the parameterisation around it.

In [8]:
tr, te = train.copy(), test.copy()
for f in (tr, te):
    f["distance_c"] = f["d"] - tr["d"].mean()
    f["weight_c"] = f["m"] - tr["m"].mean()
    f["speed_c"] = f["avg_speed_kmh"] - tr["avg_speed_kmh"].mean()

variants = {}

variants["adopted: traction + P_aux*t"] = te["pred"]

lena = smf.ols(
    "energy_kwh ~ distance_c + I(distance_c**2) + weight_c + speed_c"
    " + distance_c:weight_c",
    data=tr,
).fit()
variants["exploration candidate (centred, quad, speed)"] = lena.predict(te)

variants["schema shape: m*d*(f_w + f_s*v^2)"] = (
    sm.OLS(tr["energy_kwh"], tr[["md", "mdv2"]]).fit().predict(te[["md", "mdv2"]])
)

variants["mass-proportional only: c*m*d"] = (
    sm.OLS(tr["energy_kwh"], tr[["md"]]).fit().predict(te[["md"]])
)

variants["flat 28 kWh/km (what the backend does today)"] = 28.0 * te["d"]

rows = []
for name, pred in variants.items():
    r = (te["energy_kwh"] - pred).abs() / te["energy_kwh"] * 100
    rows.append(
        {
            "variant": name,
            "MAE": mean_absolute_error(te["energy_kwh"], pred),
            "RMSE": float(np.sqrt(((te["energy_kwh"] - pred) ** 2).mean())),
            "R2": r2_score(te["energy_kwh"], pred),
            "med_rel_%": r.median(),
        }
    )
print(pd.DataFrame(rows).round(3).to_string(index=False))

print("\nSpeed coefficient when fitted on the main sample instead of the sweep:")
print(
    f"  exploration candidate, speed_c: {lena.params['speed_c']:+.3f} kWh per km/h"
    f"   (p = {lena.pvalues['speed_c']:.4f})"
)
print(
    "  Negative, and identified from between-route variation. This is the "
    "confound 01b exists to break, not a physical result."
)

print("\nPer-composition bias, adopted form vs mass-proportional:")
mass_only = sm.OLS(data["energy_kwh"], data[["md"]]).fit()
cmp = per_comp[["weight_t", "observed_kwh_km", "bias_pct"]].copy()
cmp["mass_only_bias_pct"] = (
    data.assign(p=mass_only.predict(data[["md"]]))
    .groupby("composition_id")
    .apply(lambda x: x["p"].sum() / x["energy_kwh"].sum() - 1, include_groups=False)
    * 100
)
print(cmp.sort_values("weight_t").round(2).to_string())

                                     variant      MAE     RMSE     R2  med_rel_%
                 adopted: traction + P_aux*t  259.128  424.952  0.947     10.502
exploration candidate (centred, quad, speed)  151.367  218.123  0.986      7.591
           schema shape: m*d*(f_w + f_s*v^2)  197.504  287.354  0.976     11.290
               mass-proportional only: c*m*d  196.287  291.840  0.975     10.982
flat 28 kWh/km (what the backend does today) 4086.986 5413.270 -7.678    187.896

Speed coefficient when fitted on the main sample instead of the sweep:
  exploration candidate, speed_c: +1.470 kWh per km/h   (p = 0.0002)
  Negative, and identified from between-route variation. This is the confound 01b exists to break, not a physical result.

Per-composition bias, adopted form vs mass-proportional:
                weight_t  observed_kwh_km  bias_pct  mass_only_bias_pct
composition_id                                                         
REF-COUCH-6        313.0             6.77      0.

## 6b. Regenerate the backend coefficients module

`models/energy/calibrated_coefficients.py` is generated here and committed.
The backend imports it; nothing reads the seed CSVs at runtime. Commit the
regenerated file together with the calibration outputs - hand edits are
overwritten by the next run.

In [9]:
from datetime import date
from pathlib import Path

DRAG_UP_TO = float(
    data.loc[data["source"] == "speed", "avg_speed_kmh"].max()
    if (data["source"] == "speed").any()
    else data["avg_speed_kmh"].max()
)
P_HOTEL_KW = 15.0  # assumption - documented in the generated comments
N_ROWS = len(data)
today = date.today().isoformat()

lines = [
    "# calibrated_coefficients.py",
    "# ==========================",
    "# Fitted coefficients of the night train energy model.",
    "#",
    "# REGENERATED by calib/02_energy_calibration.ipynb - DO NOT EDIT BY HAND.",
    f"# Provenance: full 02 run of {today}, {N_ROWS} fit rows.",
    "#",
    "# Mass basis: coach gross weight at 80% load, LOCOMOTIVE EXCLUDED",
    "# (Composition.total_weight_t()); the locomotive resistance lives in the",
    "# per-km constant B. Length basis: coach rake length, locomotive excluded.",
    "",
    "from __future__ import annotations",
    "",
    f'PROVENANCE: str = "02-fit-{today}"',
    "",
    "# Highest realised average leg speed in the calibration sweep. Above it",
    "# the v^2 term extrapolates on physics (exponent verified by",
    "# exponent_check.py), not on data.",
    f"DRAG_IDENTIFIED_UP_TO_KMH: float = {DRAG_UP_TO:.1f}",
    "",
    "# Start/stop energy per leg, per tonne. Without it, legs under 50 km",
    "# under-predict by about a third.",
    f"A_PER_LEG_KWH_PER_T: float = {A:.6g}",
    "",
    "# Per-km constant, mass-independent: locomotive resistance and",
    "# everything common to every composition.",
    f"B_PER_KM_KWH: float = {B:.6g}",
    "",
    "# Rolling resistance per coach tonne-km.",
    f"C_PER_TKM_KWH: float = {Ck:.6g}",
    "",
    "# kWh/(t*km*(km/h)^2). Mass-linked share of drag.",
    f"K_MASS_DRAG: float = {K_M_DRAG:.6g}",
    "",
    "# kWh/(m*km*(km/h)^2). Length-linked share of drag (skin friction),",
    "# identified by the mass-matched composition pairs: +17% length at",
    "# equal mass costs +6-8% drag.",
    f"K_LENGTH_DRAG: float = {K_L_DRAG:.6g}",
    "",
    "# Locomotive auxiliary draw, measured against leg time (flat across",
    "# all compositions in the calibration sample).",
    f"P_LOCO_AUX_KW: float = {P_AUX_KW:.6g}",
    "",
    "# ASSUMPTION, not measured: Trassenfinder was queried with coach hotel",
    "# load off. Installed comfort power is 50-100 kVA per coach (UIC 550",
    "# era; the UIC train-line cap is 50 kVA per coach); 15 kW is a mid",
    "# estimate of the average draw, range 10-25 by season. Linear in time -",
    "# changing it never requires re-collection.",
    f"P_HOTEL_PER_COACH_KW: float = {P_HOTEL_KW}",
    "",
]

out_path = Path("..") / "calibrated_coefficients.py"
out_path.write_text("\n".join(lines), encoding="utf-8")
print(f"wrote {out_path.resolve()}")
print(
    f"  A={A:.4g}  B={B:.4g}  C={Ck:.4g}  K_m={K_M_DRAG:.4g}  "
    f"K_L={K_L_DRAG:.4g}  P_aux={P_AUX_KW:.4g}  drag<= {DRAG_UP_TO:.1f} km/h"
)

wrote C:\Users\david\PycharmProjects\energy-model\backend\models\energy\calibrated_coefficients.py
  A=0  B=1.33  C=0.001336  K_m=4.302e-07  K_L=8.713e-07  P_aux=99.92  drag<= 179.4 km/h


## 7. Export the seed

One row per coefficient, named and carrying its unit. The database reads this
file; nothing downstream should ever parse a statsmodels parameter index.

Everything under `seed/` is generated. To change a value, change this notebook.

In [10]:
n_sweep_groups = sweep.groupby(GROUP_KEYS).ngroups if speed is not None else 0

coefficients = pd.DataFrame(
    [
        {
            "coefficient": "energy_factor_base",
            "value": A,
            "unit": "kWh/t",
            "symbol": "a",
            "estimated_from": SAMPLE,
            "description": "Per-country-leg constant, scales with trailing mass",
        },
        {
            "coefficient": "energy_factor_distance",
            "value": B,
            "unit": "kWh/km",
            "symbol": "b",
            "estimated_from": SAMPLE,
            "description": "Per-km, mass-independent; includes hauling the locomotive",
        },
        {
            "coefficient": "energy_factor_weight",
            "value": Ck,
            "unit": "kWh/(t*km)",
            "symbol": "c",
            "estimated_from": SAMPLE,
            "description": "Per tonne-kilometre of trailing mass",
        },
        {
            "coefficient": "energy_factor_speed",
            "value": K_SPEED,
            "unit": "kWh/(t*km*(km/h)^2)",
            "symbol": "k",
            "estimated_from": (
                f"samples_speed ({n_sweep_groups} route-stable groups)"
                if K_SPEED
                else "not estimated - sweep not collected"
            ),
            "description": "Aerodynamic term, applied to squared average speed",
        },
        {
            "coefficient": "energy_factor_speed_length",
            "value": K_LEN_SLOPE if ADOPTED_FORM == "length" else 0.0,
            "unit": "kWh/(m*km*(km/h)^2)",
            "symbol": "k2",
            "estimated_from": (
                "samples_speed, mass-matched pairs"
                if ADOPTED_FORM == "length"
                else "not adopted - mass form fits as well or better"
            ),
            "description": ("Drag per metre of train length. Zero under the mass form"),
        },
        {
            "coefficient": "energy_factor_speed_base",
            "value": K_LEN_CONST if ADOPTED_FORM == "length" else 0.0,
            "unit": "kWh/(km*(km/h)^2)",
            "symbol": "k1",
            "estimated_from": (
                "samples_speed, mass-matched pairs"
                if ADOPTED_FORM == "length"
                else "not adopted - mass form fits as well or better"
            ),
            "description": (
                "Length-independent drag, frontal area. Zero under the mass form"
            ),
        },
        {
            "coefficient": "energy_aux_power_kw",
            "value": P_AUX_KW,
            "unit": "kW",
            "symbol": "P_aux",
            "estimated_from": (
                f"{SAMPLE} (energy split)" if P_AUX_KW else "not estimated"
            ),
            "description": (
                "Constant auxiliary draw, applied to running time. Coach hotel "
                "load excluded by the request, so this is a floor"
            ),
        },
        {
            "coefficient": "energy_factor_terrain",
            "value": 0.0,
            "unit": "kWh/(t*km) per terrain point",
            "symbol": "t",
            "estimated_from": "not estimable - Germany only, no terrain variance",
            "description": "Held at zero until Austrian and Swiss routes are collected",
        },
    ]
)

coefficients.to_csv(SEED_DIR / "energy_coefficients.csv", index=False)
print(coefficients.to_string(index=False))
print(f"\n-> {SEED_DIR / 'energy_coefficients.csv'}")

# Provenance next to the numbers, so a seeded database can be traced back.
pd.DataFrame(
    [
        {"key": "sample", "value": SAMPLE},
        {"key": "n_samples", "value": len(data)},
        {"key": "n_routes", "value": data["segment_id"].nunique()},
        {"key": "n_compositions", "value": data["composition_id"].nunique()},
        {"key": "distance_km_min", "value": round(data["d"].min(), 1)},
        {"key": "distance_km_max", "value": round(data["d"].max(), 1)},
        {
            "key": "avg_speed_kmh_p05",
            "value": round(data["avg_speed_kmh"].quantile(0.05), 1),
        },
        {
            "key": "avg_speed_kmh_p95",
            "value": round(data["avg_speed_kmh"].quantile(0.95), 1),
        },
        {"key": "mass_basis", "value": "wagenzugmasse_80pct_excl_loco"},
        {
            "key": "holdout_r2",
            "value": round(r2_score(test["energy_kwh"], test["pred"]), 4),
        },
        {"key": "holdout_median_rel_pct", "value": round(rel.median(), 2)},
        {"key": "speed_sweep_groups", "value": n_sweep_groups},
        {"key": "energy_split_available", "value": SPLIT_AVAILABLE},
        {"key": "p_aux_kw", "value": round(P_AUX_KW, 2)},
        {"key": "drag_form", "value": ADOPTED_FORM},
        {
            "key": "drag_identified_up_to_kmh",
            "value": (
                round(sweep["avg_speed_kmh"].max(), 1) if speed is not None else None
            ),
        },
        {
            "key": "bremshundertstel",
            "value": (
                sorted(data["bremshundertstel"].unique().tolist())
                if "bremshundertstel" in data.columns
                else "not recorded"
            ),
        },
        {
            "key": "streckenklasse",
            "value": (
                sorted(data["streckenklasse"].unique().tolist())
                if "streckenklasse" in data.columns
                else "not recorded"
            ),
        },
    ]
).to_csv(SEED_DIR / "energy_calibration_meta.csv", index=False)
print(f"-> {SEED_DIR / 'energy_calibration_meta.csv'}")

               coefficient        value                         unit symbol                                    estimated_from                                                                                                    description
        energy_factor_base 0.000000e+00                        kWh/t      a                                       samples_all                                                            Per-country-leg constant, scales with trailing mass
    energy_factor_distance 1.330435e+00                       kWh/km      b                                       samples_all                                                      Per-km, mass-independent; includes hauling the locomotive
      energy_factor_weight 1.335642e-03                   kWh/(t*km)      c                                       samples_all                                                                           Per tonne-kilometre of trailing mass
       energy_factor_speed 8.411943e-07          kWh

## 8. Handover to the backend

### The formula

$$E_{kWh,l} = \underbrace{a \cdot m_t + d_{km,l} \cdot
  \left( b + c \cdot m_t + k \cdot m_t \cdot \bar{v}^2_{kmh,l} \right)}_{\text{traction}}
  + \underbrace{P_{aux} \cdot t_{h,l}}_{\text{auxiliaries}}$$

Applied per country leg, which is how `route_factory.py` calls it. The two parts
are fitted separately because they respond to different things: traction to the
square of speed, auxiliaries to time. `calc.py` prices the sum.

### What the backend must get right

| | |
|---|---|
| **Mass** | `Composition.total_weight_t()` — coaches only. **Not** `total_gross_weight_t()`; the locomotive is already inside `b` (section 2). |
| **Speed** | `CountryLeg` realised average speed, the same quantity fitted here. Not the composition's `v_max`. |
| **New columns** | `composition_type_energy_factor_base`, `composition_type_energy_factor_distance` and `composition_type_energy_aux_power_kw` alongside the existing `_weight`, `_speed`, `_terrain`. |
| **Running time** | The auxiliary term needs leg running time. `CountryLeg` already carries `driving_time_min`. |
| **Existing columns** | `_weight` takes `c` directly, unit unchanged. `_speed` takes `k`. `_terrain` goes to 0.0. |
| **Fleet-wide** | The coefficients do not vary by composition type — mass does the differentiating — so every seeded composition type takes the same five values. This is why the eight calibrated compositions can seed all eleven. |

### The per-leg constant

`a * m` is applied once per country leg, so a trip crossing three countries
picks it up three times. At around 500 t that is roughly 23 kWh per leg, worth
about 4 km of running. Small enough to accept, large enough to be a written
decision rather than an accident. The alternative is applying it once per
segment, which needs a signature change in `calc_energy_consumption.py`.

### Validity range

The coefficients describe a locomotive-hauled night train on the German
network, at average leg speeds inside the range recorded in
`energy_calibration_meta.csv` as `drag_identified_up_to_kmh`. Outside that
range, and outside Germany, the model extrapolates on structure rather than
evidence — and drag is quadratic, so the error grows fast rather than
gracefully.

**The backend should flag legs that exceed it**, not silently predict them. The
target network includes French LGVs at 300-320 km/h, which Trassenfinder cannot
query at all, so those legs will be extrapolation by construction.

One further caveat that a single number cannot express: the model fits and
predicts on **average** leg speed, while drag responds to the instantaneous
speed profile. A leg averaging 130 km/h with a 230 km/h high-speed section and
slow approaches burns more than one held steady at 130. That holds only while
the profile shape resembles the calibration sample's, which is part of why the
sweep includes high-speed corridors rather than only distance-stratified
segments.

`k` is estimated over the booked-speed range of the sweep. It is a real
aerodynamic response, but it is one number fitted to a subset of segments; it
should not be read as a general drag coefficient for any train.

### Still open

- **Terrain.** Not estimable from any German source. Needs an Austrian and
  Swiss collection, which is a new route list, not a re-run.
- **Regenerative braking.** Trassenfinder's `rueckspeisung` price is set in the
  request but the returned figure is not decomposed, so recovery is inside the
  fitted constants at whatever rate the API assumes.
- **Coach hotel load.** Excluded by the request
  (`zusaetzlicher_energieverbrauch_pro_wagen_kw` is 0), so `P_aux` covers only
  the locomotive. A night train's sleeping cars draw heating, air conditioning
  and lighting all night, and none of it is in these numbers.
- **Country legs.** Every sample is station to station. The backend predicts per
  country leg, and a leg is a sub-segment. The model is additive in distance
  apart from `a * m`, so this is a known approximation rather than a bug.